In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../generated_data/subjects_AND_sampling_metadata.csv", parse_dates=["created_at"])

C:\Users\yuyue\AppData\Local\Temp\ipykernel_18008\279944522.py:4: DtypeWarning: Columns (3,14,15,16,17,18,26,27,28,29,39,41,42,46,48,49,70,71,72,73,75,76,77,78,79) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("generated_data/subjects_AND_sampling_metadata.csv", parse_dates=["created_at"])


In [2]:
# Sample about 15% of users, stratified by demographics

p_users = 0.15
np.random.seed(42) 

strata_vars = [
    "gpt_QUOTA_ETHNICITY",
    "gpt_QUOTA_AGE",
    "gpt_QUOTA_SEX",
    "gpt_QUOTA_HIGHEST EDUCATIONAL QUALIFICATION"
]

users_unique = df.drop_duplicates(subset=["id"]).copy()

users_unique["strata"] = users_unique[strata_vars].astype(str).agg("||".join, axis=1)

sampled_users = (
    users_unique.groupby("strata", group_keys=False)
    .apply(lambda x: x.sample(max(1, int(np.floor(p_users * len(x))))))
    .reset_index(drop=True)
)

sampled_user_ids = set(sampled_users["id"])

C:\Users\yuyue\AppData\Local\Temp\ipykernel_18008\499376572.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(max(1, int(np.floor(p_users * len(x))))))


In [16]:
tweets_df = df[df["id"].isin(sampled_user_ids)].copy()

In [17]:
tweets_df

,gpt_id,id,created_at.users,description,location,protected,verified_type,username,public_metrics.followers_count,public_metrics.following_count,...,gpt_speculation_QUOTA_PAYING ATTENTION TO THE 2024 PRESIDENTIAL ELECTION,gpt_speculation_QUOTA_PARTISAN LOYALTIES,"gpt_speculation_QUOTA_IDEOLOGICALLY, THIS PERSON APPEARS TO BE",gpt_speculation_QUOTA_PAST VOTE - TURNOUT IN THE 2020 PRESIDENTIAL ELECTION,gpt_speculation_QUOTA_PAST VOTE - TURNOUT IN THE 2022 HOUSE OF REPRESENTATIVES ELECTION,gpt_speculation_QUOTA_PAST VOTE - VOTE CHOICE IN THE 2020 PRESIDENTIAL ELECTION,gpt_speculation_QUOTA_PAST VOTE - VOTE CHOICE IN THE 2022 HOUSE OF REPRESENTATIVES ELECTION,clean...pool,clean...pool.date,clean...state_simple
62675,079f22ee85ae1d60084d28f1154ca0ee28b6757d406d40...,1715552550806749184,2023-10-21T02:16:28.000Z,"Strong Christian morals and values, with conse...",Montana,False,none,Delam46,151,39,...,20,10,10,30.0,30,20,20,race,2024-10-15T05:32:12Z,Montana
62676,079f22ee85ae1d60084d28f1154ca0ee28b6757d406d40...,1715552550806749184,2023-10-21T02:16:28.000Z,"Strong Christian morals and values, with conse...",Montana,False,none,Delam46,151,39,...,20,10,10,30.0,30,20,20,race,2024-10-15T05:32:12Z,Montana
62677,079f22ee85ae1d60084d28f1154ca0ee28b6757d406d40...,1715552550806749184,2023-10-21T02:16:28.000Z,"Strong Christian morals and values, with conse...",Montana,False,none,Delam46,151,39,...,20,10,10,30.0,30,20,20,race,2024-10-15T05:32:12Z,Montana
62678,079f22ee85ae1d60084d28f1154ca0ee28b6757d406d40...,1715552550806749184,2023-10-21T02:16:28.000Z,"Strong Christian morals and values, with conse...",Montana,False,none,Delam46,151,39,...,20,10,10,30.0,30,20,20,race,2024-10-15T05:32:12Z,Montana
62679,079f22ee85ae1d60084d28f1154ca0ee28b6757d406d40...,1715552550806749184,2023-10-21T02:16:28.000Z,"Strong Christian morals and values, with conse...",Montana,False,none,Delam46,151,39,...,20,10,10,30.0,30,20,20,race,2024-10-15T05:32:12Z,Montana
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2144643,fe446ad4178e9118f4c1b3db6556f2be127b4e8d88302f...,90123499,2009-11-15T08:09:19.000Z,Dulsura por mi Mami de San Juan Puerto Rico Fu...,USA,False,none,dulcebombomcito,1420,1216,...,60,65,75,80.0,80,85,85,crime,2024-10-15T05:17:33Z,USA
2144644,fe446ad4178e9118f4c1b3db6556f2be127b4e8d88302f...,90123499,2009-11-15T08:09:19.000Z,Dulsura por mi Mami de San Juan Puerto Rico Fu...,USA,False,none,dulcebombomcito,1420,1216,...,60,65,75,80.0,80,85,85,crime,2024-10-15T05:17:33Z,USA
2144645,fe446ad4178e9118f4c1b3db6556f2be127b4e8d88302f...,90123499,2009-11-15T08:09:19.000Z,Dulsura por mi Mami de San Juan Puerto Rico Fu...,USA,False,none,dulcebombomcito,1420,1216,...,60,65,75,80.0,80,85,85,crime,2024-10-15T05:17:33Z,USA
2144646,fe446ad4178e9118f4c1b3db6556f2be127b4e8d88302f...,90123499,2009-11-15T08:09:19.000Z,Dulsura por mi Mami de San Juan Puerto Rico Fu...,USA,False,none,dulcebombomcito,1420,1216,...,60,65,75,80.0,80,85,85,crime,2024-10-15T05:17:33Z,USA


In [3]:
#validating the distribution of the demographic characeteristic in the sample drawed
def compare_distributions(df_full, df_sample, vars_to_check):
    for var in vars_to_check:
        print(f"\n--- {var} ---")
        
        full_counts = df_full[var].value_counts(dropna=False)
        full_pct = full_counts / len(df_full) * 100
        
        sample_counts = df_sample[var].value_counts(dropna=False)
        sample_pct = sample_counts / len(df_sample) * 100
        
        comparison = pd.DataFrame({
            "full_count": full_counts,
            "full_pct": full_pct.round(2),
            "sample_count": sample_counts,
            "sample_pct": sample_pct.round(2)
        }).fillna(0)
        
        print(comparison)

compare_distributions(
    df_full=df.drop_duplicates(subset=["id"]),        
    df_sample=sampled_users,                         
    vars_to_check=strata_vars                        
)



--- gpt_QUOTA_ETHNICITY ---
                                                    full_count  full_pct  \
gpt_QUOTA_ETHNICITY                                                        
white - individuals with origins in any of the ...         802     86.42   
black or african american - individuals with or...          71      7.65   
hispanic or latino - includes individuals of me...          38      4.09   
asian - individuals with origins in any of the ...          10      1.08   
multiracial - individuals who identify explicit...           7      0.75   

                                                    sample_count  sample_pct  
gpt_QUOTA_ETHNICITY                                                           
white - individuals with origins in any of the ...           112       72.26  
black or african american - individuals with or...            17       10.97  
hispanic or latino - includes individuals of me...            14        9.03  
asian - individuals with origins in any of 

In [19]:
from collections import Counter
import pandas as pd
from datetime import datetime

# Define pre-election period
election_date = pd.Timestamp("2024-11-05", tz="UTC")
start_period = election_date - pd.DateOffset(years=1)
end_period = election_date
df_pre_election = tweets_df[(tweets_df["created_at"] >= start_period) & (tweets_df["created_at"] <= end_period)].copy()

# Compute user activity 
full_year_user_counts = df_pre_election['id'].value_counts(normalize=True)


In [20]:
from scipy.spatial.distance import jensenshannon
import numpy as np

window_size = pd.Timedelta(days=60)
step_size = pd.Timedelta(days=15)

start = start_period
results = []

while start + window_size <= end_period:
    end = start + window_size
    window_tweets = df_pre_election[(df_pre_election['created_at'] >= start) & 
                              (df_pre_election['created_at'] < end)]
    
    # Compute tweet count per user in window
    window_user_counts = window_tweets['id'].value_counts(normalize=True)
    
    # Align both distributions to the same user index set
    all_users = full_year_user_counts.index.union(window_user_counts.index)
    full_dist = full_year_user_counts.reindex(all_users, fill_value=0)
    window_dist = window_user_counts.reindex(all_users, fill_value=0)
    
    # Compute Jensen-Shannon Divergence
    js_div = jensenshannon(full_dist, window_dist, base=2)

    results.append({
        'start': start,
        'end': end,
        'js_divergence': js_div,
        'tweet_count': len(window_tweets),
        'user_count': window_user_counts.size
    })

    start += step_size

windows_df = pd.DataFrame(results)
best_window = windows_df.sort_values(by='js_divergence').iloc[0]

print("Most representative window:")
print(best_window)


Most representative window:
start            2024-08-16 00:00:00+00:00
end              2024-10-15 00:00:00+00:00
js_divergence                     0.281293
tweet_count                         134267
user_count                             146
Name: 19, dtype: object


In [ ]:
from datetime import datetime

# Define pre-election period
election_date = pd.Timestamp("2024-11-05", tz="UTC")
start_period = election_date - pd.DateOffset(years=1)
end_period = election_date
df_pre_election = tweets_df[(tweets_df["created_at"] >= start_period) & (tweets_df["created_at"] <= end_period)].copy()

window_days = 60  
results = []

min_date = df_pre_election["created_at"].min().normalize()
max_date = df_pre_election["created_at"].max().normalize()

for start_day in pd.date_range(min_date, max_date - pd.Timedelta(days=window_days)):
    end_day = start_day + pd.Timedelta(days=window_days)
    df_window = df_pre_election[(df_pre_election["created_at"] >= start_day) & (df_pre_election["created_at"] < end_day)]
    
    tweets_per_user = df_window.groupby("id").size()
    if len(tweets_per_user) == 0:
        continue
    
    avg_tweets = tweets_per_user.mean()
    num_users = len(tweets_per_user)
    
    results.append({
        "start": start_day,
        "end": end_day,
        "avg_tweets_per_user": avg_tweets,
        "num_users": num_users
    })

activity_df = pd.DataFrame(results)

# Get median avg tweets per user across all candidate windows
median_avg = activity_df["avg_tweets_per_user"].median()

# Rank windows by closeness to median 
activity_df["deviation"] = (activity_df["avg_tweets_per_user"] - median_avg).abs()
best_windows = activity_df.sort_values("deviation").head(10)


In [23]:
activity_df

,start,end,avg_tweets_per_user,num_users,deviation
0,2023-11-05 00:00:00+00:00,2024-01-04 00:00:00+00:00,139.934783,46,242.490217
1,2023-11-06 00:00:00+00:00,2024-01-05 00:00:00+00:00,137.851064,47,244.573936
2,2023-11-07 00:00:00+00:00,2024-01-06 00:00:00+00:00,138.212766,47,244.212234
3,2023-11-08 00:00:00+00:00,2024-01-07 00:00:00+00:00,139.276596,47,243.148404
4,2023-11-09 00:00:00+00:00,2024-01-08 00:00:00+00:00,140.531915,47,241.893085
...,...,...,...,...,...
296,2024-08-27 00:00:00+00:00,2024-10-26 00:00:00+00:00,827.205298,151,444.780298
297,2024-08-28 00:00:00+00:00,2024-10-27 00:00:00+00:00,815.642384,151,433.217384
298,2024-08-29 00:00:00+00:00,2024-10-28 00:00:00+00:00,793.450980,153,411.025980
299,2024-08-30 00:00:00+00:00,2024-10-29 00:00:00+00:00,779.188312,154,396.763312


In [26]:
best_windows 

,start,end,avg_tweets_per_user,num_users,deviation
161,2024-04-14 00:00:00+00:00,2024-06-13 00:00:00+00:00,382.425000,80,0.000000
159,2024-04-12 00:00:00+00:00,2024-06-11 00:00:00+00:00,382.544304,79,0.119304
150,2024-04-03 00:00:00+00:00,2024-06-02 00:00:00+00:00,382.597403,77,0.172403
140,2024-03-24 00:00:00+00:00,2024-05-23 00:00:00+00:00,382.616438,73,0.191438
151,2024-04-04 00:00:00+00:00,2024-06-03 00:00:00+00:00,380.896104,77,1.528896
156,2024-04-09 00:00:00+00:00,2024-06-08 00:00:00+00:00,384.000000,78,1.575000
152,2024-04-05 00:00:00+00:00,2024-06-04 00:00:00+00:00,384.441558,77,2.016558
139,2024-03-23 00:00:00+00:00,2024-05-22 00:00:00+00:00,380.342466,73,2.082534
157,2024-04-10 00:00:00+00:00,2024-06-09 00:00:00+00:00,385.256410,78,2.831410
158,2024-04-11 00:00:00+00:00,2024-06-10 00:00:00+00:00,385.410256,78,2.985256


In [ ]:
import pandas as pd
import numpy as np

def sample_non_overlapping_windows(df, user_col, time_col, window_days, k, random_state=None):

    rng = np.random.default_rng(random_state)
    
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col], utc=True)

    all_windows = []

    for uid, group in df.groupby(user_col):
        times = group[time_col].sort_values().to_list()
        if not times:
            continue

        start_time = min(times)
        end_time = max(times)
        total_days = (end_time - start_time).days + 1

        if total_days < window_days:
            continue

        # Generate all possible start dates for windows fully inside span
        possible_starts = pd.date_range(
            start=start_time.normalize(),
            end=end_time - pd.Timedelta(days=window_days - 1),
            freq="D"
        ).to_pydatetime().tolist()

        sampled_windows = []
        while len(sampled_windows) < k and possible_starts:
            start_candidate = rng.choice(possible_starts)
            candidate_end = start_candidate + pd.Timedelta(days=window_days - 1)
            sampled_windows.append((uid, start_candidate, candidate_end))

            # Remove overlapping candidates
            possible_starts = [
                d for d in possible_starts
                if (d + pd.Timedelta(days=window_days - 1) < start_candidate) or (d > candidate_end)
            ]

        all_windows.extend(sampled_windows)

    windows_df = pd.DataFrame(all_windows, columns=[user_col, "window_start", "window_end"])

    merged_df = df.merge(windows_df, on=user_col, how="left")
    merged_df["in_window"] = (
        (merged_df[time_col] >= merged_df["window_start"]) &
        (merged_df[time_col] <= merged_df["window_end"])
    )

    return merged_df, windows_df
